In [11]:
import geopandas as gpd
import pandas as pd

In [21]:
# --- Parameters ---
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
gdf = gpd.read_file(shapefile_path)



In [24]:
# Filter to only England & Wales
gdf = gdf[gdf[id_column].str[0].isin(["E", "W"])]

# Ensure consistent projection
gdf = gdf.to_crs(epsg=27700)  # British National Grid

# Compute centroids (in projected CRS)
gdf["centroid_x"] = gdf.geometry.centroid.x
gdf["centroid_y"] = gdf.geometry.centroid.y

# Build neighbor list (rook contiguity, optimized with spatial index)
rows = []
for idx, area in gdf.iterrows():
    # Use spatial index for speed
    possible_matches_index = list(gdf.sindex.intersection(area.geometry.bounds))
    possible_matches = gdf.iloc[possible_matches_index]

    # Find actual touching neighbors
    touching = possible_matches[possible_matches.geometry.touches(area.geometry)]

    # Append one row per neighbor
    for _, neighbor in touching.iterrows():
        rows.append({
            id_column: area[id_column],
            name_column: area[name_column],
            "neighbour_name": neighbor[name_column],
            "neighbour_id": neighbor[id_column],
            "centroid_x": area["centroid_x"],
            "centroid_y": area["centroid_y"]
        })

# Create DataFrame in long format
neighbors_df = pd.DataFrame(rows)

print(neighbors_df)


        LAD25CD         LAD25NM        neighbour_name neighbour_id  \
0     E06000001      Hartlepool      Stockton-on-Tees    E06000004   
1     E06000001      Hartlepool         County Durham    E06000047   
2     E06000002   Middlesbrough  Redcar and Cleveland    E06000003   
3     E06000002   Middlesbrough       North Yorkshire    E06000065   
4     E06000002   Middlesbrough      Stockton-on-Tees    E06000004   
...         ...             ...                   ...          ...   
1587  W06000023           Powys               Wrexham    W06000006   
1588  W06000023           Powys          Denbighshire    W06000004   
1589  W06000024  Merthyr Tydfil     Rhondda Cynon Taf    W06000016   
1590  W06000024  Merthyr Tydfil            Caerphilly    W06000018   
1591  W06000024  Merthyr Tydfil                 Powys    W06000023   

         centroid_x     centroid_y  
0     447873.372078  530735.890750  
1     447873.372078  530735.890750  
2     450415.357748  516581.889072  
3     45041

In [ ]:
# Optionally save
neighbors_df.to_csv("england_wales_local_authority_neighbors.csv", index=False)